In [1]:
import requests
import os
import pandas as pd
tokenCanvas = os.environ['tokenCanvas']

headers = {
    'Authorization': f'Bearer {tokenCanvas}',
}
params = {
    'per_page': 100,
    'type[]': 'StudentEnrollment',
    'state[]': 'active',
}



In [2]:
def finn_rel(link_header):
    link_header_dict = {}
    for link in link_header.split(","):
        url, rel = link.strip().split(";")
        rel = rel.split('=')[1]
        link_header_dict[rel.strip().replace('"', '')] = url.strip().replace('<', '').replace('>', '')      # Kommentar: denne er med BERRE for å skape balanse i .org-fila: \"
    return link_header_dict

# Hente alle studentar i eit emne
Dette er eit skript eg treng innimellom: korleis få ei liste over alle studentar i eit emne, for så å kopiere dei inn i eit anna emne (til dømes KBP på tvers-emna).

In [5]:
emne = 32491
url = f"https://hvl.instructure.com/api/v1/courses/{emne}/enrollments"

dr_liste = []
hentmeir = True
while hentmeir:
    respons = requests.get(url, headers=headers, params=params)
    if 200 <= respons.status_code < 300:
        data = respons.json()
        hentmeir = "next" in respons.headers['link']
        if hentmeir:
            url = finn_rel(respons.headers['link'])['next']
            print(url)   # for kontrollen sin skuld
# Hent ut data (NB! dette vil variere frå endepunkt til endepunkt)
        dataliste = []
        for element in data:
            dataliste.append([element['user']['id'], element['user']['name'], element['user']['login_id'], element['user']['sis_user_id']])
        df = pd.DataFrame(dataliste, columns=['id', 'name', 'login_id', 'sis_user_id'])
        dr_liste.append(df)
    # hentmeir = False
# og etter at eg er ferdig å lese inn slår eg dei saman:
alledata = pd.concat((df for df in dr_liste if not df.empty), ignore_index=True)


No kan eg hente ut alle "Påloggings-id" som eg kan kopiere inn i eit emne:

In [6]:
pålogging = alledata['login_id']
pålogging.to_csv('pålogging.csv', index=False, header=False)